# Step 3: Validate Outputs

**Run this AFTER both previous notebooks have finished.**

Checks performed:
1. `leetcode_clean.jsonl` — record count, empty content, content source breakdown, difficulty/category distribution
2. `company_questions.jsonl` — company count, problems per company, orphaned IDs (problems in company lists not found in clean dataset)
3. Writes a `data/my/validation_report.txt` you can keep as a reference

In [1]:
# Cell 1 — Load Both Outputs
import json
import pandas as pd
from pathlib import Path

PROJECT_ROOT  = Path(r"C:/PRAKSHIT/VS CODE/Prep Assistant Project/placed_in")
LC_FILE       = PROJECT_ROOT / "data" / "my" / "leetcode_clean.jsonl"
CO_FILE       = PROJECT_ROOT / "data" / "my" / "company_questions.jsonl"
REPORT_FILE   = PROJECT_ROOT / "data" / "my" / "validation_report.txt"

def load_jsonl(path: Path) -> list:
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

print(f"Loading {LC_FILE.name} ...")
lc_records = load_jsonl(LC_FILE)
df_lc = pd.DataFrame(lc_records)
print(f"  {len(df_lc):,} records loaded")

print(f"Loading {CO_FILE.name} ...")
co_records = load_jsonl(CO_FILE)
df_co = pd.DataFrame(co_records)
print(f"  {len(df_co):,} records loaded")

Loading leetcode_clean.jsonl ...
  3,943 records loaded
Loading company_questions.jsonl ...
  17,666 records loaded


In [2]:
# Cell 2 — Validate leetcode_clean.jsonl
lines = ["=" * 60, "VALIDATION REPORT", "=" * 60, ""]
lines.append("[1] leetcode_clean.jsonl")
lines.append("-" * 40)

# Total count
total = len(df_lc)
lines.append(f"Total problems        : {total:,}")

# Empty content check
empty_mask = (
    df_lc["content"].isna() |
    (df_lc["content"].astype(str).str.strip() == "") |
    (df_lc["content"].astype(str).str.strip() == "None")
)
empty_count = empty_mask.sum()
lines.append(f"Empty content         : {empty_count:,}  {'✅ PASS' if empty_count == 0 else '⚠️  WARN — some problems have no content'}")

# Content source
lines.append("\nContent sources:")
for src, cnt in df_lc["contentSource"].value_counts().items():
    lines.append(f"  {src:<20}: {cnt:,} ({cnt/total*100:.1f}%)")

# Difficulty breakdown
lines.append("\nDifficulty:")
for diff, cnt in df_lc["difficulty"].value_counts().items():
    lines.append(f"  {diff:<10}: {cnt:,}")

# Category breakdown
lines.append("\nCategory:")
for cat, cnt in df_lc["category"].value_counts().items():
    lines.append(f"  {cat:<20}: {cnt:,}")

# isPaidOnly
paid_count = df_lc["isPaidOnly"].sum() if "isPaidOnly" in df_lc.columns else "N/A"
lines.append(f"\nisPaidOnly = True     : {paid_count:,}")

# Spot checks
lines.append("\nSpot checks:")
r265 = df_lc[df_lc["id"] == 265]
if not r265.empty:
    src = r265.iloc[0].get("contentSource", "?")
    lines.append(f"  #265 Paint House II contentSource: {src}  {'✅' if src == 'leetcode_repo' else '❌ expected leetcode_repo'}")
else:
    lines.append("  #265 Paint House II: NOT FOUND ❌")

r1 = df_lc[df_lc["id"] == 1]
if not r1.empty:
    src = r1.iloc[0].get("contentSource", "?")
    tags = r1.iloc[0].get("topicTags", [])
    lines.append(f"  #1 Two Sum contentSource  : {src}  {'✅' if src == 'leetcode_api' else '⚠️'}")
    lines.append(f"  #1 Two Sum topicTags      : {tags}")

report_text = "\n".join(lines)
print(report_text)

VALIDATION REPORT

[1] leetcode_clean.jsonl
----------------------------------------
Total problems        : 3,943
Empty content         : 0  ✅ PASS

Content sources:
  leetcode_api        : 3,175 (80.5%)
  leetcode_repo       : 768 (19.5%)

Difficulty:
  Medium    : 2,061
  Easy      : 946
  Hard      : 936

Category:
  Algorithms          : 3,525
  Database            : 323
  JavaScript          : 67
  pandas              : 15
  Concurrency         : 9
  Shell               : 4

isPaidOnly = True     : 768

Spot checks:
  #265 Paint House II contentSource: leetcode_repo  ✅
  #1 Two Sum contentSource  : leetcode_api  ✅
  #1 Two Sum topicTags      : ['array', 'hash-table']


In [3]:
# Cell 3 — Validate company_questions.jsonl
co_lines = ["", "[2] company_questions.jsonl", "-" * 40]

total_co = len(df_co)
co_lines.append(f"Total (company, problem) pairs : {total_co:,}")

unique_companies = df_co["company"].nunique()
co_lines.append(f"Unique companies               : {unique_companies:,}")

# Per-company stats
per_company = df_co.groupby("company").size()
co_lines.append(f"Problems per company (min)     : {per_company.min()}")
co_lines.append(f"Problems per company (max)     : {per_company.max()}")
co_lines.append(f"Problems per company (avg)     : {per_company.mean():.0f}")
co_lines.append(f"Problems per company (median)  : {per_company.median():.0f}")

# Top 5 companies by problem count
co_lines.append("\nTop 5 companies by problem count:")
for company, cnt in per_company.sort_values(ascending=False).head(5).items():
    co_lines.append(f"  {company:<30}: {cnt:,}")

# Orphaned problem IDs (in company lists but not in leetcode_clean)
lc_ids = set(df_lc["id"].dropna().astype(int))
co_ids = set(df_co["problemId"].dropna().astype(int))
orphaned = co_ids - lc_ids
co_lines.append(f"\nProblem IDs in company lists but NOT in leetcode_clean : {len(orphaned)}")
if orphaned:
    co_lines.append(f"  Sample: {sorted(orphaned)[:15]}")
    co_lines.append("  (These are likely contest/SQL problems not in the JSONL)")
else:
    co_lines.append("  ✅ All company problem IDs match the clean dataset")

# Window coverage
co_lines.append("\nWindow coverage across all companies:")
for win in ["30d", "3m", "6m", "6m+", "all"]:
    count = df_co["windows"].apply(lambda ws: win in ws).sum()
    co_lines.append(f"  {win:<6}: {count:,} pairs ({count/total_co*100:.1f}%)")

co_report = "\n".join(co_lines)
print(co_report)

# Write full report to file
full_report = report_text + "\n" + co_report + "\n"
REPORT_FILE.write_text(full_report, encoding="utf-8")
print(f"\n✅ Validation report saved to:")
print(f"   {REPORT_FILE}")


[2] company_questions.jsonl
----------------------------------------
Total (company, problem) pairs : 17,666
Unique companies               : 656
Problems per company (min)     : 1
Problems per company (max)     : 2274
Problems per company (avg)     : 27
Problems per company (median)  : 4

Top 5 companies by problem count:
  google                        : 2,274
  amazon                        : 1,957
  microsoft                     : 1,374
  meta                          : 1,366
  bloomberg                     : 1,190

Problem IDs in company lists but NOT in leetcode_clean : 0
  ✅ All company problem IDs match the clean dataset

Window coverage across all companies:
  30d   : 399 pairs (2.3%)
  3m    : 1,771 pairs (10.0%)
  6m    : 3,851 pairs (21.8%)
  6m+   : 15,691 pairs (88.8%)
  all   : 17,641 pairs (99.9%)

✅ Validation report saved to:
   C:\PRAKSHIT\VS CODE\Prep Assistant Project\placed_in\data\my\validation_report.txt
